**Sections**

1. Setup
2. Mapping Tables
3. Helper Functions
4. Training Data
5. Evaluation Data
6. Final Checks
7. Upload to Hugging Face

# 1. Setup

In [13]:
%pip install -q huggingface_hub

In [14]:
import re

import pandas as pd
from datasets import Dataset, load_dataset
from huggingface_hub import login

## Configuration

In [15]:
RANDOM_STATE = 42

ARASAFE_SYNTHETIC_PATH = "/content/arasafe_synthetic_prompts.jsonl"
ARASAFE_NATURAL_PATH = "/content/arasafe_natural_prompts.jsonl"

ARABGUARD_URL = (
    "https://huggingface.co/datasets/d12o6aa/ArabGuard-Egyptian-V1"
    "/resolve/main/ArabGuard_Clean_v2.csv"
)
ODDADMIX_REPO = "oddadmix/arabic-guardrail"

SOURCE_ARABGUARD = "ArabGuard-Egyptian-V1"
SOURCE_ARASAFE_SYNTHETIC = "AraSafe-benchmark-synthetic"
SOURCE_ARASAFE_NATURAL = "AraSafe-benchmark-natural"
SOURCE_ODDADMIX = "oddadmix-arabic-guardrail"

TRAIN_REPO = "OussamaBenSlama/qarai-guard-data-v0"
EVAL_REPO = "OussamaBenSlama/qarai-guard-data-eval-v0"

TRAIN_COLUMNS = ["text", "label", "language", "source", "original_label"]
EVAL_COLUMNS = ["text", "label", "language", "original_label"]

UNKNOWN_LABEL = "UNKNOWN"
NO_ARABIC_LANGUAGE = "english_or_arabizi"

# 2. Mapping Tables

Each table converts a source label to a standard label.
A source label that is not in a table becomes `UNKNOWN`.

## ArabGuard Label Map

The table lists the standard label first.
The next cell converts it to a `source label → standard label` map.

In [16]:
ARABGUARD_LABEL_GROUPS = {
    "safe": [
        "safe",
        "social_engineering_safe_daily_life",
    ],
    "jailbreak": [
        "gaslighting_trap_manipulation",
        "prompt_leaking_exfiltration",
        "jailbreak_persona_exfiltration",
        "jailbreak_persona_harmful",
        "jailbreak_persona_offensive",
        "logical_manipulation_manipulation",
    ],
    "privacy_pii": [
        "authoritative_exfiltration",
        "authoritative_manipulation",
    ],
    "harmful": [
        "authoritative_harmful",
        "code_injection_exfiltration",
        "code_injection_harmful",
        "code_injection_manipulation",
        "code_switching_exfiltration",
        "code_switching_harmful",
        "code_switching_manipulation",
        "direct_injection_exfiltration",
        "direct_injection_harmful",
        "direct_injection_manipulation",
        "direct_injection_safe_daily_life",
        "logical_manipulation_exfiltration",
        "logical_manipulation_harmful",
        "payload_splitting_exfiltration",
        "payload_splitting_harmful",
        "payload_splitting_manipulation",
        "reverse_psychology_exfiltration",
        "reverse_psychology_harmful",
        "reverse_psychology_manipulation",
        "roleplay_exfiltration",
        "roleplay_harmful",
        "roleplay_manipulation",
        "social_engineering_exfiltration",
        "social_engineering_harmful",
        "social_engineering_manipulation",
    ],
}

ARABGUARD_LABEL_MAP = {
    source_label: standard_label
    for standard_label, source_labels in ARABGUARD_LABEL_GROUPS.items()
    for source_label in source_labels
}


## AraSafe Label Map

Both AraSafe files (synthetic and natural) use this map.
The synthetic file has no safe label. This does not cause an error.

In [17]:
ARASAFE_LABEL_MAP = {
    "آمن/غير ضار": "safe",
    "محتوى ضار آخر": "harmful",
    "إرهاب أو تطرف": "harmful",
    "أنشطة غير قانونية": "harmful",
    "عنف أو أذى": "harmful",
    "محتوى صريح": "sexual_content",
    "انتهاك خصوصية": "privacy_pii",
    "معلومات مضللة": "misinformation",
    "خطاب كراهية": "hate_harassment",
}

## Oddadmix Label Map

The training split and the held split use this map.

In [18]:
ODDADMIX_LABEL_MAP = {
    "safe": "safe",
    "safe_sensitive": "safe_sensitive",
    "privacy_pii": "privacy_pii",
    "jailbreak": "jailbreak",
    "prompt_injection": "prompt_injection",
    "self_harm": "self_harm",
    "sexual_content": "sexual_content",
    "hate_harassment": "hate_harassment",
    "misinformation": "misinformation",
    "specialized_advice": "harmful",
    "violent_weapons": "harmful",
    "nonviolent_crime": "harmful",
}

## Oddadmix Language Map

The column `variety` holds the Arabic name of the language.

In [19]:
ODDADMIX_VARIETY_MAP = {
    "العربية الفصحى": "msa",
    "عربي مختلط بالإنجليزية": "msa_code_switching",
    "اللهجة المصرية": "egyptian",
    "اللهجة الخليجية": "gulf",
    "اللهجة الشامية": "syrian",
}

# 3. Helper Functions

Each function does one job. Sections 4 and 5 call these functions for every source.

## Mapping and Language Detection

In [20]:
def map_values(series, mapping, default=UNKNOWN_LABEL):
    """Replace each value with its mapped value.
    """
    return series.map(mapping).fillna(default)


ARABIC_WORD = re.compile(r"\b[\u0600-\u06FF]+\b")
LATIN_WORD = re.compile(r"\b[A-Za-z]+\b")


def detect_language(text, arabic_variety):
    """Return the language group of a text.

    Rules:
        No Arabic word          -> "english_or_arabizi"
        Arabic and Latin words  -> "<arabic_variety>_code_switching"
        Arabic words only       -> "<arabic_variety>"

    Args:
        text: The text to check.
        arabic_variety: "egyptian" or "msa". Use the variety of the source.
    """
    text = str(text)
    has_arabic = bool(ARABIC_WORD.search(text))
    has_latin = bool(LATIN_WORD.search(text))

    if not has_arabic:
        return NO_ARABIC_LANGUAGE
    if has_latin:
        return f"{arabic_variety}_code_switching"
    return arabic_variety

## Schema, Checks and Summary

In [21]:
def standardize(df, text_col, original_label_col, label_map, language, source):
    """Convert a raw source table to the standard schema.

    Args:
        df: The raw table.
        text_col: Name of the text column in `df`.
        original_label_col: Name of the label column in `df`.
        label_map: Map from source label to standard label.
        language: A series with the language of each row (same index as `df`).
        source: Value for the "source" column.

    Returns:
        A new table with the columns in `TRAIN_COLUMNS`.
    """
    out = pd.DataFrame({
        "text": df[text_col],
        "label": map_values(df[original_label_col], label_map),
        "language": language,
        "source": source,
        "original_label": df[original_label_col],
    })
    return out[TRAIN_COLUMNS]


def check_no_unknown(df, name):
    """Stop with an error if a label or a language is `UNKNOWN`."""
    for column in ("label", "language"):
        unknown = df.loc[df[column] == UNKNOWN_LABEL]
        if not unknown.empty:
            raise ValueError(
                f"{name}: {len(unknown)} rows have {column} = {UNKNOWN_LABEL}. "
                f"Check the mapping tables. Source labels: "
                f"{unknown['original_label'].unique()[:10].tolist()}"
            )


def show_summary(df, name):
    """Print the size and the main counts of a table."""
    print(f"\n{'=' * 60}\n{name}: {len(df)} rows\n{'=' * 60}")
    print("\nLabel:")
    print(df["label"].value_counts())
    print("\nLanguage:")
    print(df["language"].value_counts())
    if "source" in df.columns:
        print("\nSource:")
        print(df["source"].value_counts())
    print("\nLabel x Language:")
    print(pd.crosstab(df["label"], df["language"]))

## Hugging Face Upload

In [22]:
def push_to_hub(df, repo_id, private=True):
    """Upload a table to a Hugging Face dataset repository.
    """
    Dataset.from_pandas(df, preserve_index=False).push_to_hub(
        repo_id, private=private
    )

# 4. Training Data

Each part loads one source and converts it to the standard schema.

## ArabGuard

In [23]:
arab_guard_raw = load_dataset("csv", data_files=ARABGUARD_URL)["train"].to_pandas()

arab_guard = standardize(
    arab_guard_raw,
    text_col="text",
    original_label_col="attack_label",
    label_map=ARABGUARD_LABEL_MAP,
    language=arab_guard_raw["text"].apply(detect_language, arabic_variety="egyptian"),
    source=SOURCE_ARABGUARD,
)
check_no_unknown(arab_guard, "ArabGuard")

## AraSafe Synthetic

In [24]:
arasafe_synthetic_raw = pd.read_json(ARASAFE_SYNTHETIC_PATH, lines=True)

arasafe_synthetic = standardize(
    arasafe_synthetic_raw,
    text_col="Prompt",
    original_label_col="Safety_Label",
    label_map=ARASAFE_LABEL_MAP,
    language=arasafe_synthetic_raw["Prompt"].apply(detect_language, arabic_variety="msa"),
    source=SOURCE_ARASAFE_SYNTHETIC,
)
check_no_unknown(arasafe_synthetic, "AraSafe synthetic")

## AraSafe Natural

This source is split in two parts:

* Rows with the label `safe` go to the training data.
* All other rows go to the evaluation data.

In [25]:
arasafe_natural_raw = pd.read_json(ARASAFE_NATURAL_PATH, lines=True)

arasafe_natural = standardize(
    arasafe_natural_raw,
    text_col="User_Prompt",
    original_label_col="Safety_Label",
    label_map=ARASAFE_LABEL_MAP,
    language=arasafe_natural_raw["User_Prompt"].apply(detect_language, arabic_variety="msa"),
    source=SOURCE_ARASAFE_NATURAL,
)
check_no_unknown(arasafe_natural, "AraSafe natural")

is_safe = arasafe_natural["label"] == "safe"
arasafe_natural_train = arasafe_natural[is_safe]
arasafe_natural_eval = arasafe_natural[~is_safe]

## Oddadmix Arabic Guardrail (train split)

In [26]:
oddadmix_ds = load_dataset(ODDADMIX_REPO)
oddadmix_train_raw = oddadmix_ds["train"].to_pandas()

oddadmix_train = standardize(
    oddadmix_train_raw,
    text_col="text",
    original_label_col="label",
    label_map=ODDADMIX_LABEL_MAP,
    language=map_values(oddadmix_train_raw["variety"], ODDADMIX_VARIETY_MAP),
    source=SOURCE_ODDADMIX,
)
check_no_unknown(oddadmix_train, "Oddadmix train")

README.md:   0%|          | 0.00/6.40k [00:00<?, ?B/s]

data/train.jsonl: reconstructing file:   0%|          |  0.00B / 91.4MB            

data/train.jsonl: downloading bytes:           |  0.00B            

eval_held.jsonl:   0%|          | 0.00/2.82M [00:00<?, ?B/s]

data/eval_unseen_dialect.jsonl: reconstructing file:   0%|          |  0.00B / 18.7MB            

...

Generating train split:   0%|          | 0/196297 [00:00<?, ? examples/s]

Generating held split:   0%|          | 0/6061 [00:00<?, ? examples/s]

Generating unseen_dialect split:   0%|          | 0/39761 [00:00<?, ? examples/s]

Generating unseen_phrasing split:   0%|          | 0/8723 [00:00<?, ? examples/s]

Generating hard split:   0%|          | 0/728 [00:00<?, ? examples/s]

## Merge into `data`

Steps:

1. Join all training parts.
2. Shuffle the rows.
3. Remove rows with no Arabic word (`english_or_arabizi`).

In [27]:
data = (
    pd.concat(
        [arab_guard, arasafe_synthetic, arasafe_natural_train, oddadmix_train],
        ignore_index=True,
    )
    .sample(frac=1, random_state=RANDOM_STATE)
)
data = data[data["language"] != NO_ARABIC_LANGUAGE].reset_index(drop=True)[TRAIN_COLUMNS]

# 5. Evaluation Data

## Oddadmix Arabic Guardrail (held split)

In [28]:
oddadmix_held_raw = oddadmix_ds["held"].to_pandas()

oddadmix_held = standardize(
    oddadmix_held_raw,
    text_col="text",
    original_label_col="label",
    label_map=ODDADMIX_LABEL_MAP,
    language=map_values(oddadmix_held_raw["variety"], ODDADMIX_VARIETY_MAP),
    source=SOURCE_ODDADMIX,
)
check_no_unknown(oddadmix_held, "Oddadmix held")

## Merge into `eval_data`

Steps:

1. Join both parts.
2. Shuffle the rows.
3. Keep the evaluation columns.

In [29]:
eval_data = (
    pd.concat([oddadmix_held, arasafe_natural_eval], ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)[EVAL_COLUMNS]
)

# 6. Final Checks

Run these cells before the upload.
The checks stop with an error if a table has a wrong schema or an `UNKNOWN` value.

In [30]:
assert list(data.columns) == TRAIN_COLUMNS
assert list(eval_data.columns) == EVAL_COLUMNS
assert not data.isna().any().any(), "data has missing values"
assert not eval_data.isna().any().any(), "eval_data has missing values"

check_no_unknown(data, "data")
check_no_unknown(eval_data, "eval_data")

show_summary(data, "Training data")
show_summary(eval_data, "Evaluation data")


Training data: 220887 rows

Label:
label
safe                47594
harmful             45171
safe_sensitive      24105
privacy_pii         18233
sexual_content      16580
self_harm           16092
hate_harassment     15717
misinformation      13557
prompt_injection    12359
jailbreak           11479
Name: count, dtype: int64

Language:
language
msa                        60021
msa_code_switching         41559
egyptian                   39978
gulf                       39222
syrian                     38991
egyptian_code_switching     1116
Name: count, dtype: int64

Source:
source
oddadmix-arabic-guardrail      196297
AraSafe-benchmark-synthetic     12264
AraSafe-benchmark-natural       10628
ArabGuard-Egyptian-V1            1698
Name: count, dtype: int64

Label x Language:
language          egyptian  egyptian_code_switching  gulf    msa  \
label                                                              
harmful               8104                      167  7857  13642   
hate_harass

# 7. Upload to Hugging Face

Both repositories are private. Log in first.

In [ ]:
login()

In [32]:
# push_to_hub(data, TRAIN_REPO)
# push_to_hub(eval_data, EVAL_REPO)